# 2. 2D floor planning with rotation: fit as many items as possible

Lecture: *Week 11 - Floor Planning*, a variant of the minimise-area model in notebook 1. The items are the pieces of the *Mums Know Best!* slide without one 2 × 1 piece: the seven items of the *Why so inefficient?* slide.

**The problem.** The floor is fixed at W × H = 4 × 5. Each of the 7 items may be placed (u = 1) or left out (u = 0), and may be rotated. Place as many items as possible without overlaps.

**How to run it.** Run the cells in order, starting with the first (it installs Pyomo and HiGHS). **Allow 2–4 minutes for the solve cell in the browser**: it shows `[*]` while HiGHS works (about 2 minutes in the automatic test). In Colab or Anaconda it takes under a minute.

**Expected result.** `Objective: 6.0`: six of the seven items fit. The 7 items have total area 21 and the floor only 20, so 7 is impossible. Several layouts are optimal, so which item is left out and where the others go can differ between runs and solvers; the objective is the same.

*Python version of the MATLAB file `5 Source Files/MATLAB/floor_planning_rotation_2d_max_items.m`, written for this repository: same data and constraints, but a smaller big-M (see the data cell) and no 10-second time limit, so the solver proves the optimum.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## Load the libraries

Pyomo builds and solves the model, NumPy holds the data and Matplotlib draws the layout.

In [ ]:
from pyomo.environ import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## The data

The item sizes and the floor size from the MATLAB file. `M` switches constraints off, as in notebook 1; here it is the smallest value that is still large enough (the comment explains why).

In [ ]:
# Data (as in the MATLAB file)
w = np.array([4,3,3,3,2,2,2])
h = np.array([1,1,1,1,2,1,1])
W, H = 4, 5      # floor size
# Big-M: the MATLAB file uses M = 9999. A very large M can cause rounding trouble in the solver, so we use
# a value that is still large enough: no switched-off constraint can need more than the floor size plus
# the largest item side, here 4 + 5 + 4 = 13. (With this repository's HiGHS, M = 9999 also gives 6 items.)
M = W + H + max(w.max(), h.max())
no_items = len(w)

## The model

The variables of notebook 1 without `v` (the floor size is fixed), plus `u[i]` = 1 if item $i$ is placed. The sizes are stored as whole numbers (`int`).

In [ ]:
# Model
model = ConcreteModel()
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)
model.w = Param(model.I, initialize=lambda model, i: int(w[i-1]))
model.h = Param(model.I, initialize=lambda model, i: int(h[i-1]))

model.x = Var(model.I, within=NonNegativeReals)      # bottom-left corner
model.y = Var(model.I, within=NonNegativeReals)
model.xij = Var(model.I, model.J, within=Binary)     # which side of each other two items are
model.yij = Var(model.I, model.J, within=Binary)
model.r = Var(model.I, within=Binary)                # 1 = item rotated 90 degrees
model.u = Var(model.I, within=Binary)                # 1 = item placed on the floor

## Objective

Count the items placed: maximise $\sum_i u_i$.

In [ ]:
# Objective: the number of items placed
model.obj = Objective(expr=sum(model.u[i] for i in model.I), sense=maximize)

## Constraints

The four non-overlap constraints of notebook 1, each with an extra term $M(2 - u_i - u_j)$: if item $i$ or item $j$ is left out, this term is at least $M$ and the constraint is switched off, because an item that is not on the floor cannot overlap anything. `width` and `height` give an item's size after rotation. The last two constraints keep every placed item inside the $W \times H$ floor (slide *Amending the constraints*, fixed floor); a left-out item is released by $M(1 - u_i)$.

In [ ]:
# Constraints: two placed items must not overlap (switched off by M if either item is left out)
def width(model, i):   # width of item i after rotation
    return (1 - model.r[i]) * model.w[i] + model.r[i] * model.h[i]

def height(model, i):  # height of item i after rotation
    return (1 - model.r[i]) * model.h[i] + model.r[i] * model.w[i]

# (xij, yij) = (0, 0): item i is LEFT of item j
def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + width(model, i) <= model.x[j] + M*(model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

# (xij, yij) = (1, 0): item j is LEFT of item i
def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + width(model, j) <= model.x[i] + M*(1 - model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

# (xij, yij) = (0, 1): item i is BELOW item j
def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + height(model, i) <= model.y[j] + M*(1 + model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

# (xij, yij) = (1, 1): item j is BELOW item i
def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + height(model, j) <= model.y[i] + M*(2 - model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

# a placed item must lie inside the W x H floor (a left-out item is released by M*(1 - u[i]))
model.box_xi = Constraint(model.I, rule=lambda model, i: model.x[i] + width(model, i) <= W + M*(1 - model.u[i]))
model.box_yi = Constraint(model.I, rule=lambda model, i: model.y[i] + height(model, i) <= H + M*(1 - model.u[i]))

## Solve

This is the slow cell (see the top of the notebook). `tee=False` hides the solver log; the cell prints the status (`optimal` means proved best) and the number of items placed.

In [ ]:
# Solve (the MATLAB original stops after 10 s: intlinprog(problem, 'MaxTime', 10); here there is no limit)
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
print("Status:", results.solver.termination_condition)
print("Objective:", value(model.obj))

## Results

List the placed items with their size, rotation and corner, and draw them inside the dashed floor outline. Binary values can come back as numbers such as 0.9999999, so the code tests `> 0.5`. A corner printed as `-0` is 0.

In [ ]:
# Draw the placed items (an item is placed / rotated when its binary is 1; test > 0.5, not truthiness)
placed = [i for i in model.I if model.u[i].value > 0.5]
print("Placed items:", placed, "of", list(model.I))
for i in placed:
    rot = model.r[i].value > 0.5
    print(f"item {i}: {w[i-1]}x{h[i-1]}{' rotated' if rot else ''} at ({model.x[i].value:g}, {model.y[i].value:g})")

fig, ax = plt.subplots()
cmap = plt.get_cmap('Pastel1')
for i in placed:
    wi, hi = (h[i-1], w[i-1]) if model.r[i].value > 0.5 else (w[i-1], h[i-1])
    ax.add_patch(patches.Rectangle((model.x[i].value, model.y[i].value), wi, hi, linewidth=1.5,
                                   edgecolor='black', facecolor=cmap(i / no_items), label=str(i)))
ax.add_patch(patches.Rectangle((0, 0), W, H, fill=False, linestyle='--', edgecolor='grey'))
ax.set_xlim(-1, W + 1); ax.set_ylim(-1, H + 1); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True); ax.legend(loc='upper right')
plt.show()

## What do you notice?

The area argument (21 > 20) shows at once that 7 items cannot fit, yet HiGHS works for about two minutes in the browser to prove that 6 is the best. Why?

<details><summary>Answer</summary>

The model does not know the area argument. It only has the non-overlap and floor constraints, and if the binary variables are allowed to be fractions (the *relaxation* the solver uses for its bounds), the big-M terms can be partly switched off and all 7 items "fit". To prove that 7 is impossible the solver has to rule out the combinations of the $2\binom{7}{2} = 42$ pair binaries, plus 7 rotation and 7 placement binaries, branch by branch: the *Why so inefficient?* slide.
</details>

## Try it

Give the model the area argument as a constraint: the placed items may not cover more than the floor. Insert a new code cell just before the *Solve* cell containing

```python
model.area = Constraint(expr=sum(model.u[i] * int(w[i-1] * h[i-1]) for i in model.I) <= W * H)
```

and choose *Run → Run All Cells*. What happens to the answer and to the solve time? Then delete that cell and try a 5 × 5 floor (`W, H = 5, 5`).

<details><summary>Answer</summary>

Still 6 items, but the solve takes under a second instead of about two minutes. With the area constraint the relaxation can place at most 6.75 items, and since the number of items is a whole number, as soon as the solver finds 6 it knows that 6 is optimal. A constraint like this, which removes no real solution but tightens the relaxation, is called a *valid inequality* (or *cut*). On a 5 × 5 floor all 7 items fit (area 21 ≤ 25), and the solve takes about a second.
</details>